In [1]:
import json
import glob

In [2]:
requested_services = dict()
resolved_services = dict()
with open("SVCB.log", "r") as file:
    for line in file:
        if "SVCB Requested Service" in line: # adas.log:2025-01-16 20:30:04.464140 [debug] SVCB Requested Service: 1111
            service = line.split(":")[-1].strip()
            host = line.split(".")[0].strip()
            if host not in requested_services:
                requested_services[host] = dict()
            if service not in requested_services[host]:
                requested_services[host][service] = 1
            else:
                requested_services[host][service] += 1
        if "SVCB Resolved Service" in line:
            service = line.split(":")[-1].strip()
            host = line.split(".")[0].strip()
            if host not in resolved_services:
                resolved_services[host] = dict()
            if service not in resolved_services[host]:
                resolved_services[host][service] = 1
            else:
                resolved_services[host][service] += 1

In [25]:
publishers = dict()
with open ("publishers.json", "r") as file:
    publishers_file = json.load(file)
    for publisher in publishers_file:
        # "4049": {
        #     "serviceId": 4049,
        #     "port": 4049,
        #     "ip": "192.168.0.5",
        #     "host": "zcFR",
        #     "mcast": "224.0.2.50"
        # },
        host = publishers_file[publisher]["host"].lower()
        service = int(publishers_file[publisher]["serviceId"])
        if host not in publishers:
            publishers[host] = []
        if service not in publishers[host]:
            publishers[host].append(service)

In [26]:
subscribers = dict()
with open ("subscribers.json", "r") as file:
    subscribers_file = json.load(file)
    for subscriber in subscribers_file:
        # "1-4049": {
        #     "serviceId": 4049,
        #     "clientId": 1,
        #     "port": 4049,
        #     "ip": "192.168.0.41",
        #     "host": "inf",
        #     "mcast": "224.0.2.50"
        # },
        host = subscribers_file[subscriber]["host"].lower()
        service = int(subscribers_file[subscriber]["serviceId"])
        client = str(subscribers_file[subscriber]["clientId"])
        if host not in subscribers:
            subscribers[host] = []
        if service not in subscribers[host]:
            subscribers[host].append(service)

In [27]:
# determine hosts with no publishers or no subscribers
hosts_with_no_publishers = []
hosts_with_no_subscribers = []
for host in publishers:
    if host not in subscribers:
        hosts_with_no_subscribers.append(host)
for host in subscribers:
    if host not in publishers:
        hosts_with_no_publishers.append(host)
print(len(hosts_with_no_publishers), "hosts with no publishers")
print(len(hosts_with_no_subscribers), "hosts with no subscribers")

1 hosts with no publishers
7 hosts with no subscribers


In [28]:
# count publishers per host
total_hosts = len(publishers) + len(hosts_with_no_publishers)
if (len(hosts_with_no_publishers) > 0):
    print ("Min publishers per host: 0")
else:
    print ("Min publishers per host: ", min([len(publishers[host]) for host in publishers]))
print ("Avg publishers per host: ", sum([len(publishers[host]) for host in publishers])/total_hosts)
print ("Max publishers per host: ", max([len(publishers[host]) for host in publishers]))

Min publishers per host: 0
Avg publishers per host:  16.307692307692307
Max publishers per host:  79


In [29]:
# count subscribers per host
total_hosts = len(subscribers) + len(hosts_with_no_subscribers)
if(len(hosts_with_no_subscribers) > 0):
    print("Min subscribers per host: 0")
else:    
    print("Min subscribers per host: ", min([len(subscribers[host]) for host in subscribers]))
print("Avg subscribers per host: ", sum([len(subscribers[host]) for host in subscribers]) / total_hosts)
print("Max subscribers per host: ", max([len(subscribers[host]) for host in subscribers]))


Min subscribers per host: 0
Avg subscribers per host:  34.46153846153846
Max subscribers per host:  131


In [33]:
# number of used services and subscribing clients per host
con_subs_per_host = dict()
con_pubs_per_host = dict()
for host in publishers:
    for service in publishers[host]:
        for sub_host in subscribers:
            if sub_host == host:
                continue
            if service in subscribers[sub_host]:
                if host not in con_subs_per_host:
                    con_subs_per_host[host] = 1
                else:
                    con_subs_per_host[host] += 1
                if sub_host not in con_pubs_per_host:
                    con_pubs_per_host[sub_host] = 1
                else:
                    con_pubs_per_host[sub_host] += 1
if (len(hosts_with_no_publishers) > 0):
    print ("Min connected subscribers per host: 0")
else:
    print ("Min connected subscribers per host: ", min([con_subs_per_host[host] for host in con_subs_per_host]))
print ("Avg connected subscribers per host: ", sum([con_subs_per_host[host] for host in con_subs_per_host])/total_hosts)
print ("Max connected subscribers per host: ", max([con_subs_per_host[host] for host in con_subs_per_host]))
if (len(hosts_with_no_subscribers) > 0):
    print ("Min connected publishers per host: 0")
else:
    print ("Min connected publishers per host: ", min([con_pubs_per_host[host] for host in con_pubs_per_host]))
print ("Avg connected publishers per host: ", sum([con_pubs_per_host[host] for host in con_pubs_per_host])/total_hosts)
print ("Max connected publishers per host: ", max([con_pubs_per_host[host] for host in con_pubs_per_host]))

Min connected subscribers per host: 0
Avg connected subscribers per host:  34.46153846153846
Max connected subscribers per host:  131
Min connected publishers per host: 0
Avg connected publishers per host:  34.46153846153846
Max connected publishers per host:  174


In [5]:
pub_apps = dict()
sub_apps = dict()
with open ("applications.log", "r") as file:
    for line in file:
        #zcrr.log:2025-01-16 20:30:08.671580 [info] Application(zcrr-4032-448, 01c0) is initialized (11, 100).
        if "initialized" not in line:
            continue
        host = line.split(".")[0].strip()
        service = line.split("(")[1].split("-")[1].strip()
        if len(line.split("(")[1].split(")")[0].split("-")) == 3:
            client = line.split("(")[1].split(")")[0].split("-")[2].strip()
            if host not in sub_apps:
                sub_apps[host] = dict()
            if service not in sub_apps[host]:
                sub_apps[host][service] = client
        else:
            client = "-1"
            if host not in pub_apps:
                pub_apps[host] = []
            if service not in pub_apps[host]:
                pub_apps[host].append(service)

In [6]:
routing_requests = dict()
with open ("requests.log", "r") as file:
    for line in file:
        #adas.log:2025-01-16 21:52:30.726285 [info] REQUEST(006e): [0457.0001:255.4294967295]
        if "client" in line or "REQUEST" not in line:
            continue
        host = line.split(".")[0].strip()
        client = line.split("(")[1].split(")")[0].strip()
        client = str(int(client, 16))
        service = line.split("[")[-1].split(".")[0].strip()
        service = str(int(service, 16))
        if host not in routing_requests:
            routing_requests[host] = dict()
        if service not in routing_requests[host]:
            routing_requests[host][service] = client
        else:
            print("Duplicate request", host, service, client)

In [7]:
routing_requests_a = dict()
with open ("requests-a.log", "r") as file:
    for line in file:
        #adas.log:2025-01-16 21:52:30.726285 [info] REQUEST(006e): [0457.0001:255.4294967295]
        if "client" in line or "REQUEST" not in line:
            continue
        host = line.split(".")[0].strip()
        client = line.split("(")[1].split(")")[0].strip()
        client = str(int(client, 16))
        service = line.split("[")[-1].split(".")[0].strip()
        service = str(int(service, 16))
        if host not in routing_requests_a:
            routing_requests_a[host] = dict()
        if service not in routing_requests_a[host]:
            routing_requests_a[host][service] = client
        else:
            print("Duplicate request A", host, service, client)

In [8]:
placeholders = dict()
with open ("placeholder.log", "r") as file:
    for line in file:
        #zcrr.log:2025-01-17 09:27:00.542677 [debug] request_svcb REQUEST SVCB, placeholder added for service=4053, instance=1, major=0, minor=0
        host = line.split(".")[0].strip()
        service = line.split("=")[1].split(",")[0].strip()
        if host not in placeholders:
            placeholders[host] = []
        if service not in placeholders[host]:
            placeholders[host].append(service)

In [9]:
# check if all requested services are resolved
for host in requested_services:
    for service in requested_services[host]:
        if host in resolved_services and service in resolved_services[host]:
            print(f"{host} {service} {requested_services[host][service]} {resolved_services[host][service]}")
        else:
            print(f"{host} {service} {requested_services[host][service]} 0")

adas 1111 1 0
adas 2114 1 0
adas 2113 1 0
adas 300 1 0
adas 2112 1 0
adas 2111 1 0
adas 1112 1 0
inf 6065 1 0
inf 3031 1 0
inf 5004 1 0
inf 5006 1 0
inf 4049 1 0
inf 4006 1 0
inf 4035 1 0
inf 6004 1 0
inf 4009 1 0
inf 3010 1 0
inf 3004 1 0
inf 6036 1 0
inf 6010 1 0
inf 6023 1 0
inf 6020 1 0
inf 3021 1 0
inf 4061 1 0
inf 4060 1 0
inf 6017 1 0
inf 3022 1 0
inf 4040 1 0
inf 6013 1 0
inf 6075 1 0
inf 4027 1 0
inf 4032 1 0
inf 4039 1 0
inf 4043 1 0
inf 3042 1 0
inf 3023 1 0
inf 3012 1 0
inf 6021 1 0
inf 3035 1 0
inf 6005 1 0
inf 6006 1 0
inf 6009 1 0
inf 6025 1 0
inf 6040 1 0
inf 3000 1 0
inf 3037 2 0
zcfl 6033 1 0
zcfl 6045 1 0
zcfl 6053 1 0
zcfl 6065 1 0
zcfl 6077 1 0
zcfl 6078 1 0
zcfl 7001 1 0
zcfl 7002 1 0
zcfl 7012 1 0
zcfl 5000 1 0
zcfl 5004 1 0
zcfl 5006 1 0
zcfl 4021 1 0
zcfl 4005 1 0
zcfl 4014 1 0
zcfl 4046 1 0
zcfl 4051 1 0
zcfl 4001 1 0
zcfl 6058 1 0
zcfl 4035 1 0
zcfl 4009 1 0
zcfl 4029 1 0
zcfl 4042 1 0
zcfl 4048 1 0
zcfl 6004 1 0
zcfl 4038 1 0
zcfl 4054 1 0
zcfl 4020 1 0
zcfl

In [10]:
# how many unique services are requested and resolved per host
for host in requested_services:
    print(f"{host} {len(requested_services[host])} {len(resolved_services[host])}")
    

KeyError: 'adas'

In [26]:
# check if hosts in requested servcices are also in subscribers^
for host in requested_services:
    if host in subscribers:
        print (f"{host} is in both requested_services and subscribers")
    else:
        print (f"{host} is in requested_services but not in subscribers")

adas is in both requested_services and subscribers
inf is in both requested_services and subscribers
zcfl is in both requested_services and subscribers
zcfr is in both requested_services and subscribers
zcrl is in both requested_services and subscribers
zcrr is in both requested_services and subscribers


In [35]:
for host in subscribers:
    # check if all subscribers requested their services
    if host in requested_services:
        for service in subscribers[host]:
            if service not in requested_services[host]:
                print(f"Not requested {host} {service} 0")
    else:
        for service in subscribers[host]:
            print(f"Not requested {host} {service} 0")

Not requested inf 4049 0
Not requested inf 3031 0
Not requested inf 4006 0
Not requested inf 4035 0
Not requested inf 3010 0
Not requested inf 4009 0
Not requested inf 3004 0
Not requested inf 5006 0
Not requested inf 6020 0
Not requested inf 6010 0
Not requested inf 6017 0
Not requested inf 3021 0
Not requested inf 4060 0
Not requested inf 4061 0
Not requested inf 3022 0
Not requested inf 6013 0
Not requested inf 4040 0
Not requested inf 6075 0
Not requested inf 3023 0
Not requested inf 3012 0
Not requested inf 3042 0
Not requested inf 6021 0
Not requested inf 4027 0
Not requested inf 3035 0
Not requested inf 4039 0
Not requested inf 6006 0
Not requested inf 3037 0
Not requested inf 6025 0
Not requested inf 6040 0
Not requested inf 6005 0
Not requested inf 3000 0
Not requested inf 4043 0
Not requested inf 6009 0
Not requested inf 4032 0
Not requested zcfl 4021 0
Not requested zcfl 4005 0
Not requested zcfl 6033 0
Not requested zcfl 6065 0
Not requested zcfl 5004 0
Not requested zcfl 4

In [12]:
for host in subscribers:
    # check if all subscribers are in the initialized sub_apps
    if host in sub_apps:
        for service in subscribers[host]:
            if service not in sub_apps[host]:
                print(f"Not initialized {host} {service} 0")
    else:
        for service in subscribers[host]:
            print(f"Not initialized {host} {service} 0")

In [13]:
for host in subscribers:
    # check if all subscribers requested their services
    if host in routing_requests:
        for service in subscribers[host]:
            if service not in routing_requests[host]:
                print(f"Not requested {host} {service} 0")
    else:
        for service in subscribers[host]:
            print(f"Not requested {host} {service} 0")

Not requested zcrr 3026 0
Not requested zcrr 4011 0
Not requested zcrr 4005 0
Not requested zcrr 5004 0
Not requested zcrr 3008 0
Not requested zcrr 4014 0
Not requested zcrr 3020 0
Not requested zcrr 4046 0
Not requested zcrr 5002 0
Not requested zcrr 4001 0
Not requested zcrr 3038 0
Not requested zcrr 3011 0
Not requested zcrr 7002 0
Not requested zcrr 3041 0
Not requested zcrr 4051 0
Not requested zcrr 4006 0
Not requested zcrr 4035 0
Not requested zcrr 4029 0
Not requested zcrr 4048 0
Not requested zcrr 4009 0
Not requested zcrr 4041 0
Not requested zcrr 7012 0
Not requested zcrr 3027 0
Not requested zcrr 4017 0
Not requested zcrr 4054 0
Not requested zcrr 3010 0
Not requested zcrr 4020 0
Not requested zcrr 3004 0
Not requested zcrr 3034 0
Not requested zcrr 4034 0
Not requested zcrr 3006 0
Not requested zcrr 5006 0
Not requested zcrr 3015 0
Not requested zcrr 3003 0
Not requested zcrr 7003 0
Not requested zcrr 3009 0
Not requested zcrr 4033 0
Not requested zcrr 4057 0
Not requeste

In [14]:
for host in subscribers:
    # check if all subscribers requested their services
    if host in routing_requests_a:
        for service in subscribers[host]:
            if service not in routing_requests_a[host]:
                print(f"Not requested {host} {service} 0")
    else:
        for service in subscribers[host]:
            print(f"Not requested {host} {service} 0")

In [22]:
for host in subscribers:
    # check if all subscribers are added as placeholders
    if host in placeholders:
        for service in subscribers[host]:
            if service not in placeholders[host]:
                print(f"Not added as placeholder {host} {service} 0")
    else:
        for service in subscribers[host]:
            print(f"Not added as placeholder {host} {service} 0")

Not added as placeholder inf 6040 0
Not added as placeholder inf 6005 0
Not added as placeholder inf 6009 0
Not added as placeholder zcrr 3018 0
Not added as placeholder zcrr 3040 0
Not added as placeholder zcrr 3029 0
Not added as placeholder zcrr 3035 0
Not added as placeholder zcrr 3002 0
Not added as placeholder zcrr 3037 0
Not added as placeholder zcrr 3019 0
Not added as placeholder zcrr 3032 0
Not added as placeholder zcrr 4058 0
Not added as placeholder zcrr 4026 0
Not added as placeholder zcrr 4018 0
Not added as placeholder zcrr 4031 0
Not added as placeholder zcrr 5001 0
Not added as placeholder zcrr 3039 0
Not added as placeholder zcrr 4037 0
Not added as placeholder zcrr 3025 0
Not added as placeholder zcrr 4002 0
Not added as placeholder zcrr 3024 0
Not added as placeholder zcrr 3007 0
Not added as placeholder zcrr 3013 0
Not added as placeholder zcrr 4030 0
Not added as placeholder zcrr 3016 0
Not added as placeholder zcrr 4008 0
Not added as placeholder zcrr 3033 0
Not 

In [18]:
# list all fils in /vsomeip-configs/* directory and add there names to the list
host_list = []
filename = glob.glob("vsomeip-configs/*")
for file in filename:
    host = file.split("/")[-1].split(".")[0]
    if host not in host_list:
        host_list.append(host)
print(host_list)

['h523', 'h362', 'h14', 'h100', 'h436', 'h535', 'h561', 'h511', 'h83', 'h582', 'h348', 'h169', 'h176', 'h136', 'h86', 'h634', 'h193', 'h647', 'h7', 'h482', 'h591', 'h259', 'h85', 'h190', 'h147', 'h632', 'h62', 'h544', 'h396', 'h596', 'h373', 'h520', 'h105', 'h36', 'h537', 'h175', 'h115', 'h258', 'h93', 'h26', 'h43', 'h304', 'h209', 'h612', 'h594', 'h330', 'h9', 'h383', 'h207', 'h581', 'h359', 'h197', 'h366', 'vsomeip-udp-mininet-publisher', 'h518', 'h391', 'h431', 'h631', 'h579', 'h616', 'h292', 'h585', 'h218', 'h638', 'h246', 'h53', 'h607', 'h534', 'h297', 'h643', 'h575', 'h310', 'h334', 'h568', 'h174', 'h74', 'h464', 'h368', 'h635', 'h18', 'h230', 'h475', 'vsomeip-udp-mininet-multihost', 'h79', 'h301', 'h623', 'h389', 'h80', 'h213', 'h223', 'h255', 'h451', 'h91', 'h461', 'h380', 'h471', 'h407', 'h335', 'h363', 'h217', 'h48', 'h376', 'h441', 'h593', 'h251', 'h323', 'h546', 'h55', 'h3', 'h660', 'h150', 'h485', 'h148', 'h344', 'h31', 'h302', 'h186', 'h51', 'h119', 'h460', 'h499', 'h589'

In [29]:
contributors = []
with open ("contribs2.log","r") as contributions:
    for line in contributions:
        host = line.split(".")[0].strip()
        if host not in contributors:
            contributors.append(host)
print(contributors)

['h100', 'h101', 'h102', 'h103', 'h104', 'h105', 'h106', 'h107', 'h108', 'h109', 'h10', 'h110', 'h111', 'h112', 'h113', 'h114', 'h115', 'h116', 'h117', 'h118', 'h119', 'h11', 'h120', 'h121', 'h122', 'h123', 'h124', 'h125', 'h126', 'h127', 'h128', 'h129', 'h12', 'h130', 'h131', 'h132', 'h133', 'h134', 'h135', 'h136', 'h137', 'h138', 'h139', 'h13', 'h140', 'h141', 'h142', 'h143', 'h144', 'h145', 'h147', 'h148', 'h149', 'h14', 'h150', 'h151', 'h152', 'h153', 'h154', 'h155', 'h156', 'h157', 'h158', 'h159', 'h15', 'h160', 'h161', 'h162', 'h163', 'h164', 'h165', 'h166', 'h167', 'h168', 'h169', 'h16', 'h170', 'h171', 'h172', 'h173', 'h174', 'h175', 'h176', 'h177', 'h178', 'h179', 'h17', 'h180', 'h181', 'h182', 'h183', 'h184', 'h185', 'h186', 'h187', 'h188', 'h189', 'h18', 'h191', 'h192', 'h193', 'h194', 'h195', 'h196', 'h197', 'h198', 'h199', 'h19', 'h1', 'h200', 'h201', 'h202', 'h203', 'h204', 'h205', 'h206', 'h207', 'h208', 'h209', 'h20', 'h210', 'h211', 'h212', 'h213', 'h214', 'h215', 'h21

In [30]:
for host in host_list:
    if host not in contributors:
        print(f"{host} is not a contributor")


h190 is not a contributor
vsomeip-udp-mininet-publisher is not a contributor
vsomeip-udp-mininet-multihost is not a contributor
h376 is not a contributor
h528 is not a contributor
h527 is not a contributor
h146 is not a contributor
h365 is not a contributor
h556 is not a contributor
h61 is not a contributor
h340 is not a contributor
h611 is not a contributor
vsomeip-udp-mininet-subscriber is not a contributor
h375 is not a contributor
h71 is not a contributor
